# Vulnerability repair baseline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kounsolas/thesis/blob/main/notebooks/vulnerability_repair_baseline.ipynb)

Baseline of the diploma thesis *Creation of a Private Security Vulnerability Repair System Using Local Language Models* (Christos Kounsolas, Aristotle University of Thessaloniki).

It uses the same input, the same Bearer scan, the same vulnerability check and the same security policies as the full pipeline, but asks the model (`stelterlab/NextCoder-32B-AWQ`) for a patch once: no self-reflection, no generated tests, no candidate ranking and no selection step. The thesis compares the full pipeline against this single-shot model.

- **Runtime:** Google Colab with a GPU of about 24 GB (the saved run used an L4); the model needs roughly 21.5 GB of GPU memory.
- **Repository:** https://github.com/kounsolas/thesis
- **Thesis (PDF, in Greek):** https://github.com/kounsolas/thesis/blob/main/docs/thesis_Kounsolas.pdf
- **Full pipeline:** [vulnerability_repair_pipeline.ipynb](https://github.com/kounsolas/thesis/blob/main/notebooks/vulnerability_repair_pipeline.ipynb)

# Instructions

- Go to the section `Store the vulnerable code in a file with the correct extension`
- Paste the code in the variable `code`
- Store the programming language of the code in the variable `language`
- Run all

- Make sure that the tool is processing the correct vulnerability, since bearer might return more than one. Check the section `Inspect the vulnerabilities found from Bearer`

# Install all the necessary packages

In [ ]:
import sys, importlib, site
!{sys.executable} -m pip install -q "transformers==4.51.3" "torch==2.6.0" autoawq accelerate
!{sys.executable} -m pip uninstall -y torchvision
importlib.invalidate_caches(); importlib.reload(site)





     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 139.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:0

<module 'site' (frozen)>

# Import the necessary libraries

In [ ]:
import json
import os
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import yaml
import torch.nn.functional as F
from pprint import pprint
from pathlib import Path
from tqdm import tqdm
import subprocess
import glob
import shutil
import re
from textwrap import dedent
from typing import Dict, List, Any
from google.colab import runtime
import time

# Run bearer on the vulnerable code

## Download Bearer

In [ ]:
%%bash
set -euo pipefail

VER="1.49.0"
ARCH="$(uname -m)"
case "$ARCH" in
  x86_64|amd64) REL_ARCH="amd64" ;;
  aarch64|arm64) REL_ARCH="arm64" ;;
  *) echo "Unsupported arch: $ARCH"; exit 1 ;;
esac

mkdir -p /content/bearer_bin
cd /content/bearer_bin

# Start fresh
rm -f bearer || true
rm -f bearer_${VER}_linux_${REL_ARCH}.tar.gz || true
rm -rf bearer_${VER}_linux_${REL_ARCH} || true

echo "Downloading bearer v${VER} for ${REL_ARCH}..."
wget -q https://github.com/Bearer/bearer/releases/download/v${VER}/bearer_${VER}_linux_${REL_ARCH}.tar.gz

echo "Extracting..."
tar -xzf bearer_${VER}_linux_${REL_ARCH}.tar.gz

# Some releases unpack to a folder; some drop the binary at top-level — handle both:
if [ -d "bearer_${VER}_linux_${REL_ARCH}" ]; then
  mv bearer_${VER}_linux_${REL_ARCH}/bearer .
fi

chmod +x bearer
./bearer version
echo "OK: bearer binary at /content/bearer_bin/bearer"


Extracting...
OK: bearer binary at /content/bearer_bin/bearer


You are running an outdated version of Bearer CLI, v2.0.1 is now available. You can find update instructions at https://docs.bearer.com/reference/installation/#updating-bearer

bearer version 1.49.0, build 7569b47eef2f1d2111e96785ddb8466f5563a9e8


## Store the vulnerable code in a file with the correct extension

In [ ]:
language = "php"

code = r"""
<?php
//The PHP code can be ignored. It's just for design purpose.
include_once('./ignore/design/design.php');
$title = 'Vsnippet #19 - Open Redirect Classic';
$design = Design(__FILE__, $title);

/**
* YesWeHack - Vulnerable Code Snippet
*/
?>

<!DOCTYPE html>
<html>
    <head>
      <title><?= $title ?></title>
    </head>
<body>
<h1><?= $title ?></h1>
<ul>
    <!-- Customer mode -->
    <li><a href="#light">☀️ Light</a></li>
    <li><a href="#dark">🌘 Dark</a></li>
</ul>
<pre>New feature is out!</pre>
<script>
setInterval( () => {
    m = (location.hash).split('#')[1] ?? 'dark';
    b = document.body.style;
    if ( m == "light" ) {
        b.backgroundColor = "#fff";
        b.color =  "#000";

    } else if ( m == "dark" ) {
        b.backgroundColor = "#000";
        b.color =  "#fff"

    } else {
        location.href = "/" + m;
    }
}, 300);
</script>
<div>
<?= $design ?>
</div>
</body>
</html>
"""

LANG_TO_EXT = {
    "javascript": "js",
    "typescript": "ts",
    "python": "py",
    "java": "java",
    "go": "go",
    "ruby": "rb",
    "php": "php",
}

def pick_ext(lang: str | None) -> str:
    if not lang:
        return "txt"   # last resort
    return LANG_TO_EXT.get(lang.strip().lower(), "txt")

vuln_code_ext = pick_ext(language)

file_name = f"app.{vuln_code_ext}"

project_dir = "/content/vscode_demo"
os.makedirs(project_dir, exist_ok=True)

file_path = os.path.join(project_dir,file_name)
with open(os.path.join(file_path), "w") as f:
    f.write(code)

os.environ["TARGET_FILE"] = file_path

print("Saved file to", file_path)
print("TARGET_FILE =", os.environ["TARGET_FILE"])

Saved file to /content/vscode_demo/app.php
TARGET_FILE = /content/vscode_demo/app.php


## Scan the vulnerable file and save the results in a .yaml file

In [ ]:
%%bash
set -u

OUTDIR="/content/sast_reports_snippet"
BIN="/content/bearer_bin/bearer"
mkdir -p "$OUTDIR"
chmod +x "$BIN" 2>/dev/null || true

TARGET_FILE="${TARGET_FILE:?TARGET_FILE not set from Python}"

YAML_OUT="$OUTDIR/vscode_demo_sast.yaml"

echo "Scanning file: $TARGET_FILE"
ls -l "$TARGET_FILE"

"$BIN" scan "$TARGET_FILE" \
  --scanner sast \
  --format yaml \
  --output "$YAML_OUT" \
  --exit-code 0

echo "Saved YAML: $YAML_OUT"
echo "YAML preview (first 60 lines):"
sed -n '1,60p' "$YAML_OUT" || true


Scanning file: /content/vscode_demo/app.php
-rw-r--r-- 1 root root 949 Mar 22 21:02 /content/vscode_demo/app.php
Saved YAML: /content/sast_reports_snippet/vscode_demo_sast.yaml
YAML preview (first 60 lines):
{}



Analyzing codebase
⠋  (38/-) [0s] 
You are running an outdated version of Bearer CLI, v2.0.1 is now available. You can find update instructions at https://docs.bearer.com/reference/installation/#updating-bearer

Loading rules
Scanning target /content/vscode_demo/app.php
 └ 100% [===============] (1/1) [2s]
Running Detectors
Generating dataflow
Evaluating rules
 └ 100% [===============] (179/179) [1s]


In [ ]:
base = "/content/sast_reports_snippet/vscode_demo_sast"
candidates = [base, base + ".yaml"]

if not os.path.exists(candidates[0]):
    if os.path.exists(candidates[1]):
        report_path = candidates[1]
    else:
        matches = sorted(glob.glob(base + "*.yaml"))
        if not matches:
            raise FileNotFoundError(f"Report not found: {base} (tried {candidates} and {base}*.yaml)")
        report_path = matches[-1]
else:
    report_path = candidates[0]

with open(report_path, "r") as f:
    data = yaml.safe_load(f) or {}

sast_critical = data.get("critical", []) or []
sast_high     = data.get("high", []) or []
sast_medium   = data.get("medium", []) or []
sast_low      = data.get("low", []) or []

print("Loaded from:", report_path)
print({
    "critical": len(sast_critical),
    "high":     len(sast_high),
    "medium":   len(sast_medium),
    "low":      len(sast_low),
})


Loaded from: /content/sast_reports_snippet/vscode_demo_sast.yaml
{'critical': 0, 'high': 0, 'medium': 0, 'low': 0}


## Create functions that help me extract context from the sast report

In [ ]:
SNIPPET_DIR = "/content/vscode_demo"
SNIPPET_CONTEXT = 15

Sast_outputs = []
Numbered_code = []
vuln_snippets_info = []

def extract_snippet(filepath, line, context=SNIPPET_CONTEXT):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    start_line = max(0, int(line) - context - 1)
    end_line = min(len(lines), int(line) + context)
    return lines[start_line:end_line], start_line + 1

def number_whole_file(filepath):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    return "\n".join(f"{i+1:3} {l.rstrip()}" for i, l in enumerate(lines))

def parse_cwe_ids(cwe_ids):
    if not cwe_ids:
        return "CWE-UNKNOWN"
    return f"CWE-{str(cwe_ids[0]).strip()}"

def process_findings(vulnerabilities, severity_label):
    made, skipped = 0, 0

    for f in (vulnerabilities or []):
        file_abs = f.get("full_filename") or f.get("filename")
        if not file_abs or not isinstance(file_abs, str):
            skipped += 1
            continue

        # make relative path if under snippet dir
        file_rel = os.path.relpath(file_abs, SNIPPET_DIR) if file_abs.startswith(SNIPPET_DIR) else file_abs

        # ensure file exists
        if not os.path.isfile(file_abs):
            candidate = os.path.join(SNIPPET_DIR, file_rel)
            if os.path.isfile(candidate):
                file_abs = candidate
            else:
                print(f"[!] File not found: {file_abs}")
                skipped += 1
                continue

        line = f.get("line_number") or f.get("parent_line_number") or 1
        try:
            line = int(line)
        except Exception:
            line = 1

        rule = f.get("rule", {}) or {}
        cwe_id = parse_cwe_ids(rule.get("cwe_ids"))
        vuln_type = rule.get("title") or "Unknown"
        description = rule.get("description") or ""

        whole_numbered = number_whole_file(file_abs)



        sast_output = {
            "vulnerability_type": vuln_type,
            "cwe": cwe_id,
            "severity": severity_label.upper(),
            "description": description,
            "line": line,
            "file_path": file_rel
        }

        vuln_snippets_info.append({
            "file_path": file_rel,
            "file_abs": file_abs,
            "lang": vuln_code_ext,
            "sast_output": sast_output,
            "numbered_code_whole": whole_numbered,
        })
        made += 1

    return made, skipped

made_c, skip_c = process_findings(sast_critical, "CRITICAL")
made_h, skip_h = process_findings(sast_high, "HIGH")
made_m, skip_m = process_findings(sast_medium, "MEDIUM")
made_l, skip_l = process_findings(sast_low, "LOW")

# --- fallback: if NOTHING was found at all ---
total_findings = len(vuln_snippets_info)
if total_findings == 0:
    target_file = os.environ.get("TARGET_FILE")  # the one you scanned
    if target_file and os.path.isfile(target_file):
        whole_numbered = number_whole_file(target_file)
        file_rel = os.path.relpath(target_file, SNIPPET_DIR) if target_file.startswith(SNIPPET_DIR) else target_file

        vuln_snippets_info.append({
            "file_path": file_rel,
            "file_abs": target_file,
            "lang": vuln_code_ext,
            "sast_output": None,
            "numbered_code_whole": whole_numbered,
        })
    else:
        print("[!] No findings and TARGET_FILE is missing/unset.")

print("Total examples:", len(vuln_snippets_info))


Total examples: 1


# Policies per CWE for the patch generation

In [ ]:
POLICIES = {
  "CWE-89": {
    "name": "SQL Injection",
  "allowed": [
  "Use parameterized/prepared queries or ORM bindings",
  "Reuse existing DB helpers",
  "Minimal, in-window changes",
  "For non-parameterizable inputs (column names, ORDER BY), whitelist values must be taken exactly from valid inputs visible in the code window — never invented"
],
"disallowed": [
  "String interpolation/concatenation in SQL",
  "Adding new database dependencies",
  "Behavior/schema changes",
  "Inventing whitelist values not present in the code window"
]
  },
  "CWE-79": {
  "name": "Cross-Site Scripting",
  "allowed": [
    "HTML body/attribute context → htmlspecialchars($val, ENT_QUOTES, 'UTF-8') or html.escape()",
    "JavaScript string context → json_encode($val) or json.dumps(val), without additional surrounding quotes",
    "URL context → urlencode() or urllib.parse.quote()",
    "ENT_QUOTES flag mandatory whenever htmlspecialchars is used",
    "textContent instead of innerHTML when no HTML rendering is needed"
  ],

  "disallowed": [
    "htmlspecialchars or html.escape applied to JavaScript string injection points",
    "json_encode or json.dumps wrapped in additional quote delimiters",
    "innerHTML as a sink for user-controlled values",
    "Blacklist or regex filtering of tags or characters as primary defense",
    "Client-side escaping as the only defense",
    "Any encoding function applied without first identifying the injection context"
  ]
},
  "CWE-22": {
  "name": "Path Traversal",
  "allowed": [
    "Use path.resolve(baseDir, userInput) to get an absolute path.",
    "Compare canonical(resolvedPath) starts-with canonical(baseDir) + path.sep before any read/write.",
    "Allowlist filenames/dirs or require that basename(resolvedPath) equals the user-provided filename when appropriate.",
    "Early reject on invalid path components (e.g. containing '..', absolute paths, drive letters on Windows).",
    "Use absolute final paths or filesystem APIs where you control the root/baseDir."
  ],
  "disallowed": [
    "Using raw user input in join/resolve or filesystem calls without a containment check.",
    "Allowing access outside baseDir or relying on process.cwd().",
    "Following untrusted symlinks without safeguards.",
    "Relying only on string replaces or regex stripping of '../' to prevent traversal.",
    "String-prefix checks on the raw input path instead of the canonical resolved path.",
    "Continuing after detecting an invalid path."
  ]
},
  "CWE-78": {
  "name": "Command Injection",
  "allowed": [
    "Whitelist of allowed argument values taken from the code window — never invented",
    "Language-native non-shell APIs that accept argument lists rather than command strings (e.g. subprocess list in Python, execFile in Node.js, escapeshellarg in PHP)",
    "Preserving existing functionality — do not remove features as the sole fix"
  ],
  "disallowed": [
    "Any API that passes user input as part of a shell command string regardless of language",
    "Blacklist or character-stripping approaches for shell input sanitization",
    "Building commands via string concatenation or interpolation with user input",
    "Removing feature functionality entirely as the sole security fix"
  ]
},
  "CWE-95": {
  "name": "Eval/Code Injection",
  "allowed": [
    "Completely remove or disable dynamic code execution for untrusted input.",
    "Treat user input strictly as data (parse, validate, or reject).",
    "Use an allowlisted mini-language or a set of predefined safe operations (no free-form JS).",
    "Replace eval-like sinks with safe, non-Turing-complete interpreters when needed.",
    "If any interpreter remains, it must not execute JavaScript and must not expose globals (require, process, fs, child_process)."
  ],
  "disallowed": [
    "Using eval, new Function, Function(), vm.runIn*, or any JS evaluator on user-controlled input.",
    "Transforming or wrapping eval (e.g., JSON.stringify/parse, templates) while still executing user input.",
    "Using regex/blacklists/timeouts as the primary defense.",
    "Exposing Node.js globals to any evaluator or sandbox.",
    "Preserving dynamic code execution solely to keep original behavior."
  ]
},
  "CWE-918": {
  "name": "Server-Side Request Forgery",
  "allowed": [
    "Use a strict allowlist of (scheme, host, optional port/path) defined server-side, not from user input.",
    "Parse the user input with a URL parser and validate scheme is http or https only.",
    "Resolve the hostname to an IP and reject any private, loopback, link-local, or metadata IP ranges.",
    "Re-validate the final URL after redirects (or disable redirects entirely for user-controlled URLs).",
    "Use an HTTP client that does not automatically forward server auth cookies or tokens.",
    "Apply conservative timeouts, method restrictions (e.g., GET only), and response size limits.",
    "Standard library URL parsing and IP address modules may be imported when required to implement validation (e.g. urllib.parse, ipaddress, socket)."
  ],
  "disallowed": [
    "Passing user-provided URLs directly into HTTP clients or fetch libraries without validation.",
    "Allowing access to localhost, 127.0.0.0/8, ::1, link-local, RFC1918 private ranges, or cloud metadata IPs.",
    "Allowing non-HTTP(S) schemes (file, gopher, ftp, etc.) for user-controlled URLs.",
    "Relying only on string prefix checks or naive hostname comparison without DNS/IP validation.",
    "Blocklist-based hostname or IP validation — a blocklist can never be complete.",
    "Following redirects from a validated URL to an unvalidated host or private IP.",
    "Forwarding server-side auth headers, cookies, or tokens to user-controlled destinations."
  ]
},
  "CWE-352": {
    "name": "Cross-Site Request Forgery",
    "allowed": [
      "CSRF tokens (per action/session)",
      "SameSite cookies or double-submit",
      "Idempotent GET-only semantics"
    ],
    "disallowed": [
      "State-changing actions without token",
      "Trusting Origin/Referer alone",
      "Per-route CSRF opt-outs"
    ]
  },
  "CWE-601": {
    "name": "Open Redirect",
    "allowed": [
      "Allow only internal paths starting with `/` or exact matches from an external allowlist",
      "Parse and canonicalize the URL before comparing (no raw string checks)",
      "Fallback to `/` or 406 on unrecognized targets (no redirect with raw input)"
    ],
    "disallowed": [
      "Redirecting directly to user-controlled input",
      "Substring/regex matching to decide if a URL is allowed",
      "Permitting dangerous or opaque schemes (`javascript:`, `data:`, protocol-relative `//`)"
    ]
  },
  "CWE-73": {
  "name": "External Control of File Name or Path",
  "allowed": [
    "Treat user input as data: allowlist expected filenames/extensions when feasible.",
    "If input selects a file under a server-controlled directory, use path.resolve(baseDir, userInput) and enforce containment: resolved starts-with baseDir + path.sep.",
    "Reject absolute paths, drive letters, and any '..' segments (POSIX + Windows separators).",
    "Fail closed: reject invalid input before filesystem access."
  ],
  "disallowed": [
    "Passing raw user input into fs.* / path.join/resolve without containment checks.",
    "Relying only on regex/string replace to remove '../' or '..\\\\'.",
    "Leaking full server filesystem paths in responses or error details (unless it's explicitly a demo).",
    "Continuing after detecting invalid path."
  ]
},
  "DEFAULT": {
    "name": "Generic Hardening",
    "allowed": [
      "Minimal, in-window fix",
      "Reuse existing helpers",
      "Validate input near the sink"
    ],
    "disallowed": [
      "New dependencies",
      "Large control-flow changes",
      "Behavior/schema changes"
    ]
  }
}


# Policies per CWE for the test generation

In [ ]:
TEST_POLICIES = {
  "CWE-89": {  # SQL Injection
    "name": "SQL Injection",
    "must_have": [
  "Happy path: valid inputs visible in the code must produce correct results",
  "SQL injection payloads in any parameter must be rejected or treated as data, not executed",
  "Invalid inputs must be handled safely without crashing",
  "Input values used in tests must be derived from the code window, not invented"
]
  },
  "CWE-79": {  # XSS
    "name": "Cross-Site Scripting",
    "must_have": [
      "Untrusted HTML/JS input must NOT appear unencoded in output/DOM.",
      "Legit content must still render correctly (no broken UI).",
      "Script payloads must NOT execute (no onload/onerror/dangerouslySetInnerHTML effects)."
    ]
  },
  "CWE-22": {
    "name": "Path Traversal",
    "must_have": [
      "Validate and canonicalize any user-supplied path BEFORE joining/resolving it or touching the filesystem.",
      "Reject absolute paths, any parent-directory segments (e.g., '..'), and leading separators; accept only plain filenames OR enforce baseDir containment.",
      "If using baseDir containment, compute a canonical absolute path and require it to start with the canonical baseDir plus a separator.",
      "On invalid paths, abort the operation with an error (early exit).",
      "Valid files within baseDir must remain accessible via canonical absolute paths or APIs that accept an explicit root/baseDir.",
      "Handle both POSIX and Windows separators; odd encodings or long paths must not crash and must remain contained within baseDir."
    ]
  },
  "CWE-78": {
  "name": "Command Injection",
  "must_have": [
    "Valid inputs visible in the code (e.g. route names, allowed methods) must still work after the fix",
    "Inputs containing shell metacharacters must be rejected — do not enumerate specific characters as the list is never complete",
    "Input values used in tests must be derived from the code window, not invented",
    "Tests must not depend on stdout, stderr, filesystem state, or environment variables"
  ]
},
  "CWE-95": {
  "name": "Eval/Code Injection",
  "must_have": [
    "User-controlled source field <SOURCE_FIELD> must NOT be executed as code (treat as data only).",
    "Patched code must not call eval, new Function, Function(), vm.runIn*, or setTimeout/setInterval with string arguments on user-controlled input.",
    "Dangerous globals and side-effects (require, process, fs, child_process) must NOT be reachable from any evaluation/sandbox.",
    "Inputs that resemble code (e.g., while(true){}, process.exit(), require('fs')...) must be treated as data (not executed) and must not hang or crash.",
    "Happy-path responses must remain well-formed (stable JSON shape) for valid data inputs.",
    "Unicode/long/special-character inputs are treated as data and handled safely."
  ]
},
  "CWE-918": {  # SSRF
    "name": "Server-Side Request Forgery",
    "must_have": [
      "User URLs to localhost/127.0.0.1/internal private ranges/metadata endpoints must be blocked and return empty/safe response.",
      "At least one valid external http or https URL must succeed (happy path) — use a clearly public domain visible in the code if available.",
      "Alternative encodings of blocked IPs (hex, decimal, IPv6) must be blocked.",
      "Oversized/slow responses are bounded without crash."
]
  },
  "CWE-352": {  # CSRF
    "name": "Cross-Site Request Forgery",
    "must_have": [
      "State-changing requests without valid CSRF token must fail.",
      "With valid token, operation succeeds (happy path).",
      "GET stays idempotent; no side-effects."
    ]
  },
  "CWE-601": {  # Open Redirect
    "name": "Open Redirect",
    "must_have": [
      "Exact allowlisted EXTERNAL URLs redirect with 302 and Location equal to the allowlisted value.",
      "INTERNAL absolute paths (start with `/`) are allowed and redirect with 302 to that path.",
      "Missing/invalid/unsafe targets (unrecognized host, dangerous scheme, protocol-relative `//`, userinfo, private/localhost IP) do NOT redirect: respond 406 or fall back safely (e.g., `/`) without stacktrace leakage.",
      "Presence of an allowlisted URL inside a query string must NOT trigger an external redirect (remain internal or reject).",
      "Location header is canonicalized and free of CR/LF; encoded/unicode tricks do not bypass checks."
    ]
  },
  "CWE-73": {
  "name": "External Control of File Name or Path",
  "must_have": [
    "Malicious path inputs (../, ..\\\\, deep/../../x) must be rejected BEFORE any filesystem access (400/403).",
    "Absolute paths (/etc/passwd, C:\\\\Windows\\\\win.ini) must be rejected (400/403).",
    "Valid in-scope files remain accessible (happy path stays 200).",
    "Expected outputs must NOT assert OS-specific error strings or full server filesystem paths.",
    "Odd encodings (%2e%2e%2f, mixed separators) must not bypass checks or crash."
  ]
},
  "DEFAULT": {
    "name": "Generic Security",
    "must_have": [
      "Happy-path behavior unchanged (baseline still passes).",
      "Malicious/edge inputs do NOT crash and do NOT grant extra privileges.",
      "Long/unicode/special-character inputs are handled safely."
    ]
  }
}


# Load Model

##  NextCoder-32B-AWQ (quantized to 4bits)
- 21.5 GB GPU RAM
- 9 min to finish patch generation

In [ ]:
model_name = "stelterlab/NextCoder-32B-AWQ"

model = AutoModelForCausalLM.from_pretrained(model_name,
                                             device_map="cuda",
                                             torch_dtype=torch.float16,
                                             trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/864 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

# Inspect the vulnerabilities found from Bearer


In [ ]:
try:
  for index, example in enumerate(vuln_snippets_info):
    print(f"{index}. {example['sast_output']['vulnerability_type']} ({example['sast_output']['severity']})")
except TypeError:
  print("SAST tool found 0 vulnerabilities")

SAST tool found 0 vulnerabilities


In [ ]:
# Always pick the first entry (either a real finding or the fallback)
vuln_snippet_info = vuln_snippets_info[0]

file_path = vuln_snippet_info.get("file_path", "")
lang = vuln_snippet_info.get("lang")
numbered_code = vuln_snippet_info["numbered_code_whole"]

sast_output = vuln_snippet_info.get("sast_output",None)  # dict or None

if sast_output is not None:
    # Normal case: we have a SAST finding
    vuln_line = sast_output.get("line")
    win_start = sast_output.get("window_start")
    win_end = sast_output.get("window_end")
    cwe = sast_output.get("cwe", "CWE-UNKNOWN")

    policy = POLICIES.get(cwe, POLICIES["DEFAULT"])
    test_policy = TEST_POLICIES.get(cwe, TEST_POLICIES["DEFAULT"])
else:
    # Fallback case: no findings
    vuln_line = None
    win_start = None
    win_end = None
    cwe = "CWE-UNKNOWN"
    policy = POLICIES["DEFAULT"]
    test_policy = TEST_POLICIES["DEFAULT"]
    sast_output = ""


print("Language:", lang)
print("CWE:", cwe)
print("Vulnerability line:", vuln_line)
print("Window start:", win_start)
print("Window end:", win_end)
print(f"Policy name: {policy['name']}")
print(f"Test policy name: {test_policy['name']}")
print(f"\n\nWhole Numbered Code: \n{numbered_code}")



Language: php
CWE: CWE-UNKNOWN
Vulnerability line: None
Window start: None
Window end: None
Policy name: Generic Hardening
Test policy name: Generic Security


Whole Numbered Code: 
  1 
  2 <?php
  3 //The PHP code can be ignored. It's just for design purpose.
  4 include_once('./ignore/design/design.php');
  5 $title = 'Vsnippet #19 - Open Redirect Classic';
  6 $design = Design(__FILE__, $title);
  7 
  8 /**
  9 * YesWeHack - Vulnerable Code Snippet
 10 */
 11 ?>
 12 
 13 <!DOCTYPE html>
 14 <html>
 15     <head>
 16       <title><?= $title ?></title>
 17     </head>
 18 <body>
 19 <h1><?= $title ?></h1>
 20 <ul>
 21     <!-- Customer mode -->
 22     <li><a href="#light">☀️ Light</a></li>
 23     <li><a href="#dark">🌘 Dark</a></li>
 24 </ul>
 25 <pre>New feature is out!</pre>
 26 <script>
 27 setInterval( () => {
 28     m = (location.hash).split('#')[1] ?? 'dark';
 29     b = document.body.style;
 30     if ( m == "light" ) {
 31         b.backgroundColor = "#fff";
 32         b.

## The model decides if the vulnerability is TP

In [ ]:
def build_tp_prompt() -> str:
    window_note = ""
    if isinstance(win_start, int) and isinstance(win_end, int):
        window_note = (
            f"\nThe snippet shows only lines {win_start}-{win_end} from the original file; "
            "line numbers are original."
        )

    has_sast = isinstance(sast_output, dict) and len(sast_output) > 0
    sast_flagged_line = vuln_line if isinstance(vuln_line, int) else None

    sast_type = None
    if has_sast:
        sast_type = (
            sast_output.get("vulnerability_type")
            or sast_output.get("title")
            or (sast_output.get("rule") or {}).get("title")
        )

    if has_sast:
        context_block = dedent(f"""\
        ## SAST report (JSON)
        {json.dumps(sast_output, indent=2)}
        SAST vulnerability type (if any): {sast_type}
        Flagged line (if any): {sast_flagged_line}
        {window_note}
        """)
        task_block = dedent("""\
        ## Task
        You must evaluate FOUR things:

        A) Existence: Does the code contain a real security vulnerability?
        B) Type correctness: Is the SAST vulnerability type correct for this code?
           - If wrong or incomplete, you must provide the correct type.
        C) CWE: Provide the best matching CWE in the form "CWE-<number>" (e.g., "CWE-89").
        D) Location: What are the correct vulnerable line numbers?
           - If you agree with the SAST flagged line (if provided), set line_match: true and include it in vuln_lines.
           - If you disagree or the flagged line is missing/invalid, set line_match: false and output corrected vuln_lines.

        Be strict and code-specific. Do not guess. If there isn’t enough evidence, set has_vulnerability: false.
        """)
        line_match_rule = (
            "Set line_match: true only if the SAST flagged line (if provided) matches the vulnerable operation "
            "(sink/source or inseparable part). Otherwise set line_match: false."
        )
        vuln_lines_rule = (
            "If has_vulnerability is false, vuln_lines MUST be []. "
            "If has_vulnerability is true and line_match is true and a flagged line exists, vuln_lines MUST include it. "
            "If has_vulnerability is true and line_match is false, vuln_lines MUST contain your corrected vulnerable lines."
        )
        type_rule = (
            "Set sast_type_correct: true only if the SAST vulnerability type matches what the code actually does. "
            "If false, set vulnerability_type to the correct type."
        )
        cwe_rule = (
            'If has_vulnerability is true, set cwe to the best matching CWE string like "CWE-89". '
            'If has_vulnerability is false, set cwe to "CWE-UNKNOWN".'
        )
    else:
        context_block = dedent(f"""\
        ## SAST report
        None (no findings were reported).
        {window_note}
        """)
        task_block = dedent("""\
        ## Task
        1) Decide if the code contains any real security vulnerability.
        2) If yes, identify the vulnerability type.
        3) If yes, provide the best matching CWE in the form "CWE-<number>" (e.g., "CWE-79").
        4) Output the exact vulnerable line numbers.
        5) Give a short, code-specific reason.

        Since there is no SAST report, you MUST set line_match: false and sast_type_correct: false.
        """)
        line_match_rule = "Because there is no SAST report, you MUST set line_match: false."
        vuln_lines_rule = (
            "If has_vulnerability is false, vuln_lines MUST be []. "
            "If has_vulnerability is true, vuln_lines MUST contain the vulnerable line numbers you identified."
        )
        type_rule = (
            "Because there is no SAST report, you MUST set sast_type_correct: false. "
            'If has_vulnerability is false, set vulnerability_type to "none".'
        )
        cwe_rule = (
            'Because there is no SAST report: if has_vulnerability is true, set cwe to the best matching "CWE-<number>"; '
            'if has_vulnerability is false, set cwe to "CWE-UNKNOWN".'
        )

    prompt = dedent(f"""\
    You are given a numbered code snippet. A SAST report may or may not be provided.
    Your job is to determine whether a real vulnerability exists, whether the SAST type is correct, the best CWE, and where it occurs.

    ## Code
    ```{lang}
    {numbered_code.strip()}
    ```

    {context_block}

    {task_block}

    ## Output Format (YAML)
    Return a YAML object matching this schema (no extra text):

    ```python
    class VulnGate(BaseModel):
        has_vulnerability: bool
        line_match: bool
        vuln_lines: List[int]         # original source line numbers from the snippet
        sast_type_correct: bool
        vulnerability_type: str       # if no vuln -> "none"
        cwe: str                      # e.g., "CWE-89"
        reason: str                   # ≤ 3 sentences, code-specific
    ```

    Rules:
    - If has_vulnerability is false:
      - vuln_lines MUST be []
      - vulnerability_type MUST be "none"
      - cwe MUST be "CWE-UNKNOWN"
    - {line_match_rule}
    - {vuln_lines_rule}
    - {type_rule}
    - {cwe_rule}
    - vuln_lines must refer to the line numbers shown in the snippet.

    Return only the YAML object.
    """)
    return prompt

In [ ]:
tp_prompt = build_tp_prompt()
print(tp_prompt)

    You are given a numbered code snippet. A SAST report may or may not be provided.
    Your job is to determine whether a real vulnerability exists, whether the SAST type is correct, the best CWE, and where it occurs.

    ## Code
    ```php
    1 
  2 <?php
  3 //The PHP code can be ignored. It's just for design purpose.
  4 include_once('./ignore/design/design.php');
  5 $title = 'Vsnippet #19 - Open Redirect Classic';
  6 $design = Design(__FILE__, $title);
  7 
  8 /**
  9 * YesWeHack - Vulnerable Code Snippet
 10 */
 11 ?>
 12 
 13 <!DOCTYPE html>
 14 <html>
 15     <head>
 16       <title><?= $title ?></title>
 17     </head>
 18 <body>
 19 <h1><?= $title ?></h1>
 20 <ul>
 21     <!-- Customer mode -->
 22     <li><a href="#light">☀️ Light</a></li>
 23     <li><a href="#dark">🌘 Dark</a></li>
 24 </ul>
 25 <pre>New feature is out!</pre>
 26 <script>
 27 setInterval( () => {
 28     m = (location.hash).split('#')[1] ?? 'dark';
 29     b = document.body.style;
 30     if ( m == "l

In [ ]:
messages = [
    {"role": "system", "content": "Return only valid YAML. No code fences, no prose."},
    {"role": "user", "content":tp_prompt}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# --- Generate
with torch.no_grad():
    output = model.generate(
        **model_inputs,
        do_sample=True,  #FALSE = the model does greedy decoding (it picks the most likely token every time)
        max_new_tokens=512,  #300 truncated the output
        temperature=0.9,
        return_dict_in_generate=True,
        output_scores=False
    )

# --- Extract output ids and text
output_ids = output.sequences[0][model_inputs.input_ids.shape[-1]:]
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True).strip()


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print(generated_text)

has_vulnerability: true
line_match: false
vuln_lines:
  - 39
sast_type_correct: false
vulnerability_type: Open Redirect
cwe: CWE-601
reason: The code constructs a URL using the `location.hash` value without proper validation or sanitization, allowing an attacker to redirect users to arbitrary URLs.


In [ ]:
answer = yaml.safe_load(generated_text)
if not answer["has_vulnerability"]:
  print("Model finds the code safe. ")
  while True:
    ok = input("Go to the next example. OK (y)?")
    if ok == "y":
      break
else:
  print("the code is vulnerable.\n")
  if not answer["sast_type_correct"]:
    print("The vulnerability type is wrong. Replace it with the correct one")
    sast_output = "Missing"
    print(f"The correct vulnerability type is: {answer['vulnerability_type']}")
    cwe = answer['cwe']
    print(f"CWE: {cwe}\n")
  else:
    #sast type is correct
    print("the sast type is correct")
  if not answer["line_match"]:
    print("The vulnerability line number is wrong. Replace it with the correct ones")
    vuln_line = answer["vuln_lines"]
    print(f"The vulnerable lines are: {vuln_line}")
  else:
    # vuln lines from sast are correct
    print("the vuln lines from SAST are correct")
    vuln_line = answer["vuln_lines"]
    print(f"The vulnerable lines are: {vuln_line}")


the code is vulnerable.

The vulnerability type is wrong. Replace it with the correct one
The correct vulnerability type is: Open Redirect
CWE: CWE-601

The vulnerability line number is wrong. Replace it with the correct ones
The vulnerable lines are: [39]


In [ ]:
def build_numbered_window(filepath, vuln_lines, context):
    if not vuln_lines:
        return None, None, None  # no snippet

    center = torch.median(torch.tensor(vuln_lines))
    print(f"========= FOR DEBUG ===========: \n{center}")
    snippet_lines, snippet_start = extract_snippet(filepath, center, context)
    numbered = "\n".join(
        f"{snippet_start + i:3} {l.rstrip()}"
        for i, l in enumerate(snippet_lines)
    )
    snippet_end = snippet_start + len(snippet_lines) - 1
    return numbered, snippet_start, snippet_end


file_abs = vuln_snippet_info["file_abs"]
numbered_window, win_start, win_end = build_numbered_window(file_abs, vuln_line, context=20)

policy = POLICIES.get(cwe, POLICIES["DEFAULT"])
test_policy = TEST_POLICIES.get(cwe, TEST_POLICIES["DEFAULT"])





========= FOR DEBUG ===========: 
39


In [ ]:
print("Language:", lang)
print("CWE:", cwe)
print("Vulnerability line:", vuln_line)
print("Window start:", win_start)
print("Window end:", win_end)
print(f"Policy name: {policy['name']}")
print(f"Test policy name: {test_policy['name']}")


Language: php
CWE: CWE-601
Vulnerability line: [39]
Window start: 19
Window end: 47
Policy name: Open Redirect
Test policy name: Open Redirect


In [ ]:
print(f"\n\nNumbered Window: \n{numbered_window}\n\n")
pprint(f"Policy: {policy}")
print("")
pprint(f"Test Policy: {test_policy}")



Numbered Window: 
 19 <h1><?= $title ?></h1>
 20 <ul>
 21     <!-- Customer mode -->
 22     <li><a href="#light">☀️ Light</a></li>
 23     <li><a href="#dark">🌘 Dark</a></li>
 24 </ul>
 25 <pre>New feature is out!</pre>
 26 <script>
 27 setInterval( () => {
 28     m = (location.hash).split('#')[1] ?? 'dark';
 29     b = document.body.style;
 30     if ( m == "light" ) {
 31         b.backgroundColor = "#fff";
 32         b.color =  "#000";
 33 
 34     } else if ( m == "dark" ) {
 35         b.backgroundColor = "#000";
 36         b.color =  "#fff"
 37 
 38     } else {
 39         location.href = "/" + m;
 40     }
 41 }, 300);
 42 </script>
 43 <div>
 44 <?= $design ?>
 45 </div>
 46 </body>
 47 </html>


("Policy: {'name': 'Open Redirect', 'allowed': ['Allow only internal paths "
 "starting with `/` or exact matches from an external allowlist', 'Parse and "
 "canonicalize the URL before comparing (no raw string checks)', 'Fallback to "
 "`/` or 406 on unrecognized targets (no re

# Create the prompt for the patch generation

In [ ]:
def build_yaml_prompt() -> str:

    window_note = ""
    if isinstance(win_start, int) and isinstance(win_end, int):
        window_note = f"\n(The snippet shows only lines {win_start}-{win_end} of the original file; line numbers are original.)"

    prompt_template = dedent("""
## Static Analysis Report
{sast_output}

## Vulnerable Code (numbered)
{lang}:
{numbered_window}
Note: The most likely vulnerable behavior is near **line {vuln_line}**.{window_note}

## Security Policy — {policy_name}
Allowed:
{allowed_list}
Disallowed:
{disallowed_list}

## HARD CONSTRAINTS (generic anti-hallucination)
- Do NOT add third-party dependencies not already imported
- Standard library imports (urllib, ipaddress, socket) may be added when strictly required by the security policy
- Do NOT invent helpers (e.g., sanitize*, canonical*, validate*, parse*). If helper behavior is needed, inline it using only core {lang} primitives (strings/arrays/objects/regex/URL if native).
- Must touch {vuln_line} and may replace a small contiguous block within [{win_start}, {win_end}].
- Preserve existing control flow and existing calls unless narrowing behavior is strictly required by tests/policy.
- Patch must compile/run with no undefined identifiers and no new dependencies.

## DEFENSIVE CODING RULES (generic)
- Do not assume data shapes: check types before method calls (e.g., Array.isArray(x) before x.some / x.includes; typeof x === "string" before string ops).
- Prefer primitive loops/predicates (for/for-of/some/every + equality/prefix/suffix/regex) over introducing new helpers.
- Avoid throwing new errors; prefer safe fallbacks consistent with tests/policy.
- Keep changes minimal: smallest code edit that satisfies tests and security intent.

## OUTPUT FORMAT (YAML — no markdown fences, no prose)
patch:
  replace_lines: [start_line_number, end_line_number]
  code: |
    <your secure replacement code here>
  why_it_works: "One or two sentences on why the patch prevents the vulnerability."
  complexity: "low"  # or "medium", "high"

Respond only with the YAML object above. No extra commentary or markdown.

## Repeat your task:
Output a valid YAML patch object that fixes the vulnerability under these constraints.
""")

    return prompt_template.format(
        sast_output=sast_output,
        numbered_window = numbered_window,
        vuln_line = vuln_line,
        lang = lang,
        window_note=window_note,
        policy_name = policy["name"],
        allowed_list="\n".join(f"- {a}" for a in policy.get("allowed", [])),
        disallowed_list="\n".join(f"- {d}" for d in policy.get("disallowed", [])),
        win_start=win_start,
        win_end=win_end
    )


In [ ]:
yaml_prompt = build_yaml_prompt()
print(yaml_prompt)


## Static Analysis Report
Missing

## Vulnerable Code (numbered)
php:
 19 <h1><?= $title ?></h1>
 20 <ul>
 21     <!-- Customer mode -->
 22     <li><a href="#light">☀️ Light</a></li>
 23     <li><a href="#dark">🌘 Dark</a></li>
 24 </ul>
 25 <pre>New feature is out!</pre>
 26 <script>
 27 setInterval( () => {
 28     m = (location.hash).split('#')[1] ?? 'dark';
 29     b = document.body.style;
 30     if ( m == "light" ) {
 31         b.backgroundColor = "#fff";
 32         b.color =  "#000";
 33 
 34     } else if ( m == "dark" ) {
 35         b.backgroundColor = "#000";
 36         b.color =  "#fff"
 37 
 38     } else {
 39         location.href = "/" + m;
 40     }
 41 }, 300);
 42 </script>
 43 <div>
 44 <?= $design ?>
 45 </div>
 46 </body>
 47 </html>
Note: The most likely vulnerable behavior is near **line [39]**.
(The snippet shows only lines 19-47 of the original file; line numbers are original.)

## Security Policy — Open Redirect
Allowed:
- Allow only internal paths starti

# Generation Loop


In [ ]:
messages = [
    {"role": "system", "content": "Return only valid YAML. No code fences, no prose."},
    {"role": "user", "content":yaml_prompt}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# --- Generate
with torch.no_grad():
    output = model.generate(
        **model_inputs,
        do_sample=True,  #FALSE = the model does greedy decoding (it picks the most likely token every time)
        max_new_tokens=1024,  #300 truncated the output
        temperature=0.9,
        return_dict_in_generate=True,
        output_scores=False
    )

# --- Extract output ids and text
output_ids = output.sequences[0][model_inputs.input_ids.shape[-1]:]
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True).strip()


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


In [ ]:
print(generated_text)

patch:
  replace_lines: [39, 40]
  code: |
    if (in_array($m, ['light', 'dark'])) {
        location.href = "/" + $m;
    }
  why_it_works: "This patch restricts the allowed internal paths to 'light' or 'dark' to prevent open redirects."
  complexity: "low"


# End RunTime

In [ ]:
# runtime.unassign()